# Week 2 — Feature Engineering
**Date:** March 2026  
**Goal:** Build all player, venue, and matchup features that power the 6 modules  
**Input:** master_ipl.csv  
**Output:** features_players.csv, features_venues.csv, features_matchups.csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the master dataset
master = pd.read_csv('../data/master_ipl.csv')
matches = pd.read_csv('../data/matches_clean.csv')

# Fix date columns
master['date'] = pd.to_datetime(master['date'])
matches['date'] = pd.to_datetime(matches['date'])

print("Data loaded!")
print(f"Master shape: {master.shape}")
print(f"Matches shape: {matches.shape}")

Data loaded!
Master shape: (278205, 32)
Matches shape: (1146, 15)


In [3]:
master['is_four'] = (master['batsman_runs'] == 4).astype(int)
master['is_six'] = (master['batsman_runs'] == 6).astype(int)

print("Columns added!")
print(f"Total fours: {master['is_four'].sum()}")
print(f"Total sixes: {master['is_six'].sum()}")

Columns added!
Total fours: 32113
Total sixes: 14353


In [4]:
batting_stats = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'batsman', 'date', 'season', 'batting_team'])
                 .agg(
                     runs=('batsman_runs', 'sum'),
                     balls_faced=('batsman_runs', 'count'),
                     fours=('is_four', 'sum'),
                     sixes=('is_six', 'sum'),
                     dismissed=('is_wicket', 'max')
                 ).reset_index())

# Calculate strike rate
batting_stats['strike_rate'] = (batting_stats['runs'] / 
                                 batting_stats['balls_faced'] * 100).round(2)

# Sort by player and date — critical for rolling features
batting_stats = batting_stats.sort_values(['batsman', 'date']).reset_index(drop=True)

print("Per-match batting stats built!")
print(f"Shape: {batting_stats.shape}")
print(f"\nSample — Virat Kohli last 5 matches:")
print(batting_stats[batting_stats['batsman'] == 'V Kohli'].tail(5)[
    ['date', 'runs', 'balls_faced', 'strike_rate', 'dismissed']
].to_string())

Per-match batting stats built!
Shape: (17336, 11)

Sample — Virat Kohli last 5 matches:
            date  runs  balls_faced  strike_rate  dismissed
16304 2025-05-03    62           33       187.88          1
16305 2025-05-23    43           25       172.00          1
16306 2025-05-27    54           30       180.00          1
16307 2025-05-29    12           12       100.00          1
16308 2025-06-03    43           35       122.86          1


In [5]:
# Sort by date and ball so everything is in chronological order
master = master.sort_values(['date', 'matchId', 'inning', 'over', 'ball']).reset_index(drop=True)

# Get match-level batting stats per player (runs scored per match)
batting_match = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'date', 'batsman'])
                 .agg(
                     runs=('batsman_runs', 'sum'),
                     balls_faced=('batsman_runs', 'count'),
                     fours=('is_four', 'sum'),
                     sixes=('is_six', 'sum')
                 ).reset_index())

# Calculate strike rate per match
batting_match['strike_rate'] = (batting_match['runs'] / batting_match['balls_faced'] * 100).round(2)

# Sort chronologically per player
batting_match = batting_match.sort_values(['batsman', 'date']).reset_index(drop=True)

print("Match-level batting stats created!")
print(f"Shape: {batting_match.shape}")
print(f"\nSample — Virat Kohli last 5 matches:")
print(batting_match[batting_match['batsman'] == 'V Kohli'].tail(5)[['date', 'runs', 'balls_faced', 'strike_rate']])

Match-level batting stats created!
Shape: (17633, 8)

Sample — Virat Kohli last 5 matches:
            date  runs  balls_faced  strike_rate
16586 2025-05-03    62           33       187.88
16587 2025-05-23    43           25       172.00
16588 2025-05-27    54           30       180.00
16589 2025-05-29    12           12       100.00
16590 2025-06-03    43           35       122.86


In [6]:
# Function to calculate exponential decay weighted average
def exp_decay_avg(series, decay=0.85):
    """More recent matches weighted higher. decay=0.85 means each match 
    is worth 85% of the next more recent match"""
    weights = np.array([decay ** i for i in range(len(series))])
    weights = weights[::-1]  # reverse so most recent has highest weight
    return np.average(series, weights=weights)

# Calculate rolling features for each player
# We use shift(1) to avoid data leakage (only use PAST matches to predict current)
batting_features = []

for player, group in batting_match.groupby('batsman'):
    group = group.sort_values('date').reset_index(drop=True)
    
    for i in range(len(group)):
        past = group.iloc[:i]  # only matches BEFORE current
        
        row = {
            'matchId': group.iloc[i]['matchId'],
            'batsman': player,
            'date': group.iloc[i]['date'],
            # Basic rolling stats
            'matches_played': len(past),
            'career_avg': past['runs'].mean() if len(past) > 0 else 0,
            'career_sr': past['strike_rate'].mean() if len(past) > 0 else 0,
            # Last 5 matches
            'avg_last5': past['runs'].tail(5).mean() if len(past) >= 2 else 0,
            'sr_last5': past['strike_rate'].tail(5).mean() if len(past) >= 2 else 0,
            # Last 10 matches
            'avg_last10': past['runs'].tail(10).mean() if len(past) >= 2 else 0,
            'sr_last10': past['strike_rate'].tail(10).mean() if len(past) >= 2 else 0,
            # Exponential decay form score
            'form_score': exp_decay_avg(past['runs'].values) if len(past) >= 2 else 0,
            'form_sr': exp_decay_avg(past['strike_rate'].values) if len(past) >= 2 else 0,
            # Consistency (lower std = more consistent)
            'consistency': past['runs'].std() if len(past) >= 2 else 0,
            # Peak score
            'peak_score': past['runs'].max() if len(past) > 0 else 0,
            # Boundary rate
            'boundary_rate': (past['fours'].sum() + past['sixes'].sum()) / past['balls_faced'].sum() if len(past) > 0 and past['balls_faced'].sum() > 0 else 0,
        }
        batting_features.append(row)

batting_features_df = pd.DataFrame(batting_features)

print("Batting features created!")
print(f"Shape: {batting_features_df.shape}")
print(f"\nSample features for V Kohli:")
print(batting_features_df[batting_features_df['batsman'] == 'V Kohli'].tail(3)[
    ['date', 'matches_played', 'career_avg', 'avg_last5', 'form_score', 'consistency']
].to_string())

Batting features created!
Shape: (17633, 15)

Sample features for V Kohli:
            date  matches_played  career_avg  avg_last5  form_score  consistency
16588 2025-05-27             256   33.445312       59.8   48.821026    27.446864
16589 2025-05-29             257   33.525292       56.0   49.597872    27.423194
16590 2025-06-03             258   33.441860       44.4   43.958191    27.402578


In [8]:
# How does each batsman perform in each phase?
phase_batting = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'date', 'batsman', 'phase'])
                 .agg(
                     runs=('batsman_runs', 'sum'),
                     balls=('batsman_runs', 'count')
                 ).reset_index())

phase_batting['sr'] = (phase_batting['runs'] / phase_batting['balls'] * 100).round(2)

# Pivot so each phase becomes a column
phase_pivot = phase_batting.groupby(['batsman', 'phase']).agg(
    avg_runs=('runs', 'mean'),
    avg_sr=('sr', 'mean')
).round(2).unstack(level='phase')

# Flatten column names
phase_pivot.columns = ['_'.join(col).strip() for col in phase_pivot.columns]
phase_pivot = phase_pivot.reset_index()
phase_pivot.columns = [col.replace(' ', '_') for col in phase_pivot.columns]

print("Phase batting features created!")
print(f"Shape: {phase_pivot.shape}")
print(f"Columns: {list(phase_pivot.columns)}")
print(f"\nTop 5 Death over batsmen (avg runs):")
if 'avg_runs_Death' in phase_pivot.columns:
    print(phase_pivot.nlargest(5, 'avg_runs_Death')[['batsman', 'avg_runs_Death', 'avg_sr_Death']].to_string())

Phase batting features created!
Shape: (704, 7)
Columns: ['batsman', 'avg_runs_Death', 'avg_runs_Middle', 'avg_runs_Powerplay', 'avg_sr_Death', 'avg_sr_Middle', 'avg_sr_Powerplay']

Top 5 Death over batsmen (avg runs):
           batsman  avg_runs_Death  avg_sr_Death
397  Misbah-ul-Haq           40.00        266.67
219        HM Amla           27.00        210.00
448    PC Valthaty           27.00        207.69
306      KS Bharat           26.00        200.00
111      BJ Rohrer           24.33        268.88


In [9]:
# Match-level bowling stats per bowler
bowling_match = (master[master['is_wide'] == 0]
                 .groupby(['matchId', 'date', 'bowler'])
                 .agg(
                     wickets=('is_wicket', 'sum'),
                     runs_conceded=('total_runs', 'sum'),
                     balls_bowled=('total_runs', 'count'),
                     dot_balls=('total_runs', lambda x: (x == 0).sum())
                 ).reset_index())

# Calculate economy and dot ball %
bowling_match['overs_bowled'] = bowling_match['balls_bowled'] / 6
bowling_match['economy'] = (bowling_match['runs_conceded'] / bowling_match['overs_bowled']).round(2)
bowling_match['dot_pct'] = (bowling_match['dot_balls'] / bowling_match['balls_bowled'] * 100).round(2)

# Sort chronologically
bowling_match = bowling_match.sort_values(['bowler', 'date']).reset_index(drop=True)

# Rolling bowling features
bowling_features = []

for player, group in bowling_match.groupby('bowler'):
    group = group.sort_values('date').reset_index(drop=True)
    
    for i in range(len(group)):
        past = group.iloc[:i]
        
        row = {
            'matchId': group.iloc[i]['matchId'],
            'bowler': player,
            'date': group.iloc[i]['date'],
            'career_wickets': past['wickets'].sum() if len(past) > 0 else 0,
            'career_economy': past['economy'].mean() if len(past) > 0 else 0,
            'economy_last5': past['economy'].tail(5).mean() if len(past) >= 2 else 0,
            'wickets_last5': past['wickets'].tail(5).sum() if len(past) >= 2 else 0,
            'economy_last10': past['economy'].tail(10).mean() if len(past) >= 2 else 0,
            'wickets_last10': past['wickets'].tail(10).sum() if len(past) >= 2 else 0,
            'dot_pct_career': past['dot_pct'].mean() if len(past) > 0 else 0,
            'dot_pct_last5': past['dot_pct'].tail(5).mean() if len(past) >= 2 else 0,
            'form_economy': exp_decay_avg(past['economy'].values) if len(past) >= 2 else 0,
            'form_wickets': exp_decay_avg(past['wickets'].values) if len(past) >= 2 else 0,
        }
        bowling_features.append(row)

bowling_features_df = pd.DataFrame(bowling_features)

print("Bowling features created!")
print(f"Shape: {bowling_features_df.shape}")
print(f"\nSample — JJ Bumrah last 3 matches:")
print(bowling_features_df[bowling_features_df['bowler'] == 'JJ Bumrah'].tail(3)[
    ['date', 'career_wickets', 'career_economy', 'economy_last5', 'wickets_last5', 'dot_pct_last5']
].to_string())

Bowling features created!
Shape: (13845, 13)

Sample — JJ Bumrah last 3 matches:
           date  career_wickets  career_economy  economy_last5  wickets_last5  dot_pct_last5
5006 2025-05-26             201        7.152324           5.57             12         54.498
5007 2025-05-30             202        7.144266           4.82             12         59.498
5008 2025-06-01             203        7.141528           5.12              9         56.166


In [10]:
second_innings = master[master['inning'] == 2].copy()

# Get target score for each match (first innings total)
first_innings_totals = (master[master['inning'] == 1]
                        .groupby('matchId')['total_runs']
                        .sum()
                        .reset_index())
first_innings_totals.columns = ['matchId', 'target']
first_innings_totals['target'] = first_innings_totals['target'] + 1  # need 1 more to win

# Merge target into second innings
second_innings = second_innings.merge(first_innings_totals, on='matchId', how='left')

# Calculate cumulative runs scored so far in innings
second_innings = second_innings.sort_values(['matchId', 'inning', 'over', 'ball'])
second_innings['cumulative_runs'] = second_innings.groupby('matchId')['total_runs'].cumsum()
second_innings['cumulative_wickets'] = second_innings.groupby('matchId')['is_wicket'].cumsum()

# Balls remaining (T20 = 120 balls total)
second_innings['ball_number'] = second_innings.groupby('matchId').cumcount() + 1
second_innings['balls_remaining'] = 120 - second_innings['ball_number']

# Required runs
second_innings['runs_required'] = second_innings['target'] - second_innings['cumulative_runs']

# Required run rate (per ball)
second_innings['required_run_rate'] = (
    second_innings['runs_required'] / 
    second_innings['balls_remaining'].clip(lower=1)
)

# Wickets in hand (10 - wickets fallen)
second_innings['wickets_in_hand'] = 10 - second_innings['cumulative_wickets']

# PRESSURE INDEX: high RRR + few wickets in hand = high pressure
second_innings['pressure_index'] = (
    second_innings['required_run_rate'] * 
    (1 + (10 - second_innings['wickets_in_hand']) / 10)
).round(4)

# Clip extreme values
second_innings['pressure_index'] = second_innings['pressure_index'].clip(0, 10)

print("Pressure Index created!")
print(f"\nPressure index stats:")
print(second_innings['pressure_index'].describe().round(3))
print(f"\nSample high pressure moments:")
high_pressure = second_innings[second_innings['pressure_index'] > 4][
    ['matchId', 'over', 'ball', 'runs_required', 'balls_remaining', 
     'wickets_in_hand', 'pressure_index']
].head(5)
print(high_pressure.to_string())

Pressure Index created!

Pressure index stats:
count    133903.000
mean          2.495
std           2.071
min           0.000
25%           1.400
50%           1.820
75%           2.638
max          10.000
Name: pressure_index, dtype: float64

Sample high pressure moments:
    matchId  over  ball  runs_required  balls_remaining  wickets_in_hand  pressure_index
50   335982     7     5            185               69                5          4.0217
51   335982     7     6            185               68                5          4.0809
52   335982     8     1            185               67                5          4.1418
53   335982     8     2            185               66                4          4.4848
54   335982     8     3            185               65                4          4.5538


In [12]:
venue_features = (master[master['inning'] == 1]
                  .groupby(['venue', 'matchId'])['total_runs']
                  .sum()
                  .reset_index()
                  .groupby('venue')
                  .agg(
                      avg_first_innings=('total_runs', 'mean'),
                      std_first_innings=('total_runs', 'std'),
                      matches_at_venue=('total_runs', 'count')
                  ).round(2).reset_index())

# Chasing win rate per venue
chasing_wins = matches.copy()
chasing_wins['chased_successfully'] = (
    chasing_wins.apply(lambda x: 1 if x['winner'] == x['team2'] 
                       or (x['toss_winner'] == x['winner'] 
                           and x['toss_decision'] == 'field') 
                       else 0, axis=1)
)
venue_chase = (chasing_wins.groupby('venue')['chased_successfully']
               .agg(['mean', 'count'])
               .reset_index())
venue_chase.columns = ['venue', 'chase_win_rate', 'matches']
venue_chase['chase_win_rate'] = venue_chase['chase_win_rate'].round(3)

# Merge venue features
venue_features = venue_features.merge(venue_chase[['venue', 'chase_win_rate']], 
                                       on='venue', how='left')

# Venue difficulty rating (normalized 0-1, higher = harder to bat)
venue_features['venue_difficulty'] = (
    1 - (venue_features['avg_first_innings'] - venue_features['avg_first_innings'].min()) /
    (venue_features['avg_first_innings'].max() - venue_features['avg_first_innings'].min())
).round(3)

# Filter venues with enough matches
venue_features = venue_features[venue_features['matches_at_venue'] >= 5]

print("Venue features created!")
print(f"Shape: {venue_features.shape}")
print(f"\nTop 5 easiest batting venues:")
print(venue_features.nlargest(5, 'avg_first_innings')[
    ['venue', 'avg_first_innings', 'chase_win_rate', 'venue_difficulty']
].to_string())
print(f"\nTop 5 hardest batting venues:")
print(venue_features.nsmallest(5, 'avg_first_innings')[
    ['venue', 'avg_first_innings', 'chase_win_rate', 'venue_difficulty']
].to_string())

Venue features created!
Shape: (52, 6)

Top 5 easiest batting venues:
                                                               venue  avg_first_innings  chase_win_rate  venue_difficulty
19          Himachal Pradesh Cricket Association Stadium, Dharamsala             208.80           0.200             0.206
1                                        Arun Jaitley Stadium, Delhi             200.91           0.455             0.291
40  Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh             197.80           0.400             0.325
15                                             Eden Gardens, Kolkata             196.59           0.455             0.338
44              Rajiv Gandhi International Stadium, Uppal, Hyderabad             192.50           0.556             0.382

Top 5 hardest batting venues:
                                               venue  avg_first_innings  chase_win_rate  venue_difficulty
36                                          Newlands         

In [14]:
bowler_type_map = {
    # Known fast bowlers
    'JJ Bumrah': 'pace', 'B Kumar': 'pace', 'Mohammed Shami': 'pace',
    'UT Yadav': 'pace', 'IS Sodhi': 'spin', 'YS Chahal': 'spin',
    'R Ashwin': 'spin', 'RA Jadeja': 'spin', 'SP Narine': 'spin',
    'PP Chawla': 'spin', 'A Mishra': 'spin', 'SL Malinga': 'pace',
    'DJ Bravo': 'pace', 'DW Steyn': 'pace', 'MM Sharma': 'pace',
    'P Kumar': 'pace', 'Z Khan': 'pace', 'A Nehra': 'pace',
    'MG Johnson': 'pace', 'JP Faulkner': 'pace', 'KH Pandya': 'pace',
    'HH Pandya': 'pace', 'BCJ Cutting': 'pace', 'M Morkel': 'pace',
    'K Rabada': 'pace', 'T Natarajan': 'pace', 'Arshdeep Singh': 'pace',
    'Mohammed Siraj': 'pace', 'AJ Tye': 'pace', 'SN Thissen': 'pace',
    'Rashid Khan': 'spin', 'Kuldeep Yadav': 'spin', 'AU Rashid': 'spin',
    'Washington Sundar': 'spin', 'Varun Chakravarthy': 'spin',
}

master['bowler_type'] = master['bowler'].map(bowler_type_map).fillna('pace')

# Batsman performance vs pace and spin
matchup = (master[master['is_wide'] == 0]
           .groupby(['batsman', 'bowler_type'])
           .agg(
               runs=('batsman_runs', 'sum'),
               balls=('batsman_runs', 'count'),
               dismissals=('is_wicket', 'sum')
           ).reset_index())

matchup['strike_rate'] = (matchup['runs'] / matchup['balls'] * 100).round(2)
matchup['avg'] = (matchup['runs'] / matchup['dismissals'].clip(lower=1)).round(2)

# Pivot to get pace and spin side by side
matchup_pivot = matchup.pivot(index='batsman', columns='bowler_type', 
                               values=['runs', 'strike_rate', 'avg'])
matchup_pivot.columns = ['_'.join(col) for col in matchup_pivot.columns]
matchup_pivot = matchup_pivot.reset_index()

# Weakness identifier: if SR vs spin < SR vs pace by 20+ points = spin weakness
matchup_pivot['pace_sr'] = matchup_pivot.get('strike_rate_pace', 0)
matchup_pivot['spin_sr'] = matchup_pivot.get('strike_rate_spin', 0)
matchup_pivot['spin_weakness'] = (matchup_pivot['pace_sr'] - matchup_pivot['spin_sr'] > 20).astype(int)
matchup_pivot['pace_weakness'] = (matchup_pivot['spin_sr'] - matchup_pivot['pace_sr'] > 20).astype(int)

print("Matchup matrix created!")
print(f"Shape: {matchup_pivot.shape}")
print(f"\nBatsmen with spin weakness (sample):")
spin_weak = matchup_pivot[matchup_pivot['spin_weakness'] == 1][
    ['batsman', 'pace_sr', 'spin_sr']].head(8)
print(spin_weak.to_string())

Matchup matrix created!
Shape: (704, 11)

Batsmen with spin weakness (sample):
          batsman  pace_sr  spin_sr
1        A Badoni   146.02   101.71
7        A Kamboj   123.08     0.00
8        A Kumble    79.49    50.00
11       A Mishra    92.52    55.56
12       A Mithun   145.00    83.33
14        A Nehra    67.21     0.00
15       A Nortje   106.82    40.00
16  A Raghuvanshi   146.78   120.00


In [16]:
player_dna = batting_features_df.copy()

# Merge phase features
player_dna = player_dna.merge(phase_pivot, on='batsman', how='left')

# Merge matchup features
player_dna = player_dna.merge(
    matchup_pivot[['batsman', 'pace_sr', 'spin_sr', 'spin_weakness', 'pace_weakness']], 
    on='batsman', how='left')

# Merge venue features (avg across all venues as a proxy)
venue_avg = venue_features[['avg_first_innings', 'venue_difficulty']].mean()

# Add pressure performance - how does batsman perform in high pressure?
pressure_batting = (second_innings[second_innings['is_wide'] == 0]
                    .groupby('batsman')
                    .apply(lambda x: x[x['pressure_index'] > 3]['batsman_runs'].mean())
                    .reset_index())
pressure_batting.columns = ['batsman', 'high_pressure_avg']
pressure_batting['high_pressure_avg'] = pressure_batting['high_pressure_avg'].fillna(0).round(2)

player_dna = player_dna.merge(pressure_batting, on='batsman', how='left')
player_dna['high_pressure_avg'] = player_dna['high_pressure_avg'].fillna(0)

# Fill any remaining NaN values
player_dna = player_dna.fillna(0)

print("Player DNA Vector created!")
print(f"Shape: {player_dna.shape}")
print(f"\nDNA features per player:")
print(list(player_dna.columns))
print(f"\nVirat Kohli's latest DNA profile:")
kohli = player_dna[player_dna['batsman'] == 'V Kohli'].tail(1)
print(kohli[['career_avg', 'avg_last5', 'form_score', 'consistency',
             'avg_runs_Powerplay', 'avg_runs_Death', 
             'pace_sr', 'spin_sr', 'high_pressure_avg']].to_string())

Player DNA Vector created!
Shape: (17633, 26)

DNA features per player:
['matchId', 'batsman', 'date', 'matches_played', 'career_avg', 'career_sr', 'avg_last5', 'sr_last5', 'avg_last10', 'sr_last10', 'form_score', 'form_sr', 'consistency', 'peak_score', 'boundary_rate', 'avg_runs_Death', 'avg_runs_Middle', 'avg_runs_Powerplay', 'avg_sr_Death', 'avg_sr_Middle', 'avg_sr_Powerplay', 'pace_sr', 'spin_sr', 'spin_weakness', 'pace_weakness', 'high_pressure_avg']

Virat Kohli's latest DNA profile:
       career_avg  avg_last5  form_score  consistency  avg_runs_Powerplay  avg_runs_Death  pace_sr  spin_sr  high_pressure_avg
16590    33.44186       44.4   43.958191    27.402578               16.37           15.57   134.38   123.05               1.46


In [17]:
player_dna.to_csv('../data/player_dna.csv', index=False)
batting_features_df.to_csv('../data/batting_features.csv', index=False)
bowling_features_df.to_csv('../data/bowling_features.csv', index=False)
venue_features.to_csv('../data/venue_features.csv', index=False)
matchup_pivot.to_csv('../data/matchup_matrix.csv', index=False)
second_innings[['matchId', 'batsman', 'over', 'ball', 
                'pressure_index', 'runs_required', 
                'balls_remaining', 'wickets_in_hand']].to_csv(
                '../data/pressure_index.csv', index=False)

print("All feature files saved!")
print("\nFiles created:")
print("  → player_dna.csv          (26-dim player DNA vector)")
print("  → batting_features.csv    (rolling batting stats)")
print("  → bowling_features.csv    (rolling bowling stats)")
print("  → venue_features.csv      (venue difficulty ratings)")
print("  → matchup_matrix.csv      (batsman vs bowler type)")
print("  → pressure_index.csv      (ball-by-ball pressure scores)")
print(f"\nTotal features built: 26 batting + 12 bowling + 6 venue + 4 matchup + 1 pressure = 49 features")
print("Ready for model building!")

All feature files saved!

Files created:
  → player_dna.csv          (26-dim player DNA vector)
  → batting_features.csv    (rolling batting stats)
  → bowling_features.csv    (rolling bowling stats)
  → venue_features.csv      (venue difficulty ratings)
  → matchup_matrix.csv      (batsman vs bowler type)
  → pressure_index.csv      (ball-by-ball pressure scores)

Total features built: 26 batting + 12 bowling + 6 venue + 4 matchup + 1 pressure = 49 features
Ready for model building!
